## Sportmonks Germany 201 — 01: Fixture corpus

Build the match list for **Germany** under the competitive calendar: **Euro 2024** (`league_id` 1326), **2025 UEFA Nations League** (1538), and **WC Qualification Europe** (720). Keep **2024–2025** kickoffs only and **finished** matches (`state_id` 5). Friendlies are excluded by competition whitelist.

**Run Jupyter from the repository root** so `load_dotenv(".env")` matches the other Sportmonks notebooks.

In [1]:
import json
import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
if (ROOT / "sportmonks").is_dir():
    SPORTMONKS_DIR = ROOT / "sportmonks"
    sys.path.insert(0, str(SPORTMONKS_DIR))
else:
    SPORTMONKS_DIR = ROOT
    sys.path.insert(0, str(ROOT))

import sportmonks_germany_api as smg

In [2]:
load_dotenv(".env")

API_TOKEN = os.getenv("API_TOKEN")

### Resolve Germany `team_id`
Set `GERMANY_TEAM_ID_OVERRIDE` if `/teams/search/Germany` returns multiple clubs and the heuristic picks the wrong row.

In [3]:
GERMANY_TEAM_ID_OVERRIDE = None  # e.g. 12345

team = None
hits = []
if GERMANY_TEAM_ID_OVERRIDE is not None:
    GERMANY_TEAM_ID = int(GERMANY_TEAM_ID_OVERRIDE)
else:
    hits = smg.search_teams(API_TOKEN, "Germany")

pd.DataFrame(hits).head(20) if hits else None

,id,sport_id,country_id,venue_id,gender,name,short_code,image_path,founded,type,placeholder,last_played_at
0,18660,1,11,1944.0,male,Germany,GER,https://cdn.sportmonks.com/images/soccer/teams...,1900.0,national,False,2025-11-17 19:45:00
1,59598,1,11,1944.0,male,Germany U17,GER,https://cdn.sportmonks.com/images/soccer/teams...,1900.0,national,False,2026-02-18 11:00:00
2,59622,1,11,NaN,male,Germany U19,GER U19,https://cdn.sportmonks.com/images/soccer/teams...,NaN,national,False,2025-10-14 15:00:00
3,60068,1,11,1944.0,male,Germany U21,GER U21,https://cdn.sportmonks.com/images/soccer/teams...,1900.0,national,False,2025-11-18 17:00:00
4,63338,1,11,NaN,male,Germany U23,GER U23,https://cdn.sportmonks.com/images/soccer/teams...,NaN,national,False,NaN
5,63350,1,11,NaN,female,Germany W,GER,https://cdn.sportmonks.com/images/soccer/teams...,1900.0,national,False,2026-03-07 17:00:00
6,135750,1,11,1944.0,male,Germany U20,GER U20,https://cdn.sportmonks.com/images/soccer/teams...,1900.0,national,False,2025-11-17 14:00:00
7,145350,1,11,NaN,male,Germany U18,GER U18,https://cdn.sportmonks.com/images/soccer/teams...,0.0,national,False,2025-03-23 18:00:00


In [4]:
if GERMANY_TEAM_ID_OVERRIDE is None:
    team = smg.pick_germany_national_team(hits)
    if team is None:
        raise RuntimeError("No team from search; set GERMANY_TEAM_ID_OVERRIDE")
    GERMANY_TEAM_ID = int(team["id"])

GERMANY_TEAM_ID, (team or {}).get("name")

(18660, 'Germany')

In [5]:
meta = {
    "germany_team_id": GERMANY_TEAM_ID,
    "team_name": (team or {}).get("name"),
    "league_ids": sorted(smg.LEAGUE_IDS),
}
(SPORTMONKS_DIR / "germany_201_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
meta

{'germany_team_id': 18660,
 'team_name': 'Germany',
 'league_ids': [720, 1326, 1538]}

### Ingest: chunked `fixtures/between/{start}/{end}/{team_id}`
Sportmonks limits each between-query window (use ≤100 days per chunk).

In [6]:
start = date(2024, 1, 1)
end = date(2025, 12, 31)

all_fixtures = []
for a, b in smg.date_chunks(start, end, max_days=100):
    chunk = smg.fetch_fixtures_between_team(
        API_TOKEN, GERMANY_TEAM_ID, a.isoformat(), b.isoformat()
    )
    all_fixtures.extend(chunk)

len(all_fixtures)

25

In [7]:
filtered = smg.filter_germany_corpus(all_fixtures)
df = pd.json_normalize(filtered).drop_duplicates(subset=["id"], keep="first")
df["starting_at"] = pd.to_datetime(df["starting_at"], errors="coerce")
df = df.sort_values("starting_at")
df[["id", "league_id", "season_id", "name", "starting_at", "state_id"]].tail(12)

,id,league_id,season_id,name,starting_at,state_id
8,19085369,1538,23188,Germany vs Bosnia and Herzegovina,2024-11-16 19:45:00,5
9,19085372,1538,23188,Hungary vs Germany,2024-11-19 19:45:00,5
10,19330542,1538,23188,Italy vs Germany,2025-03-20 19:45:00,5
11,19330543,1538,23188,Germany vs Italy,2025-03-23 19:45:00,5
12,19330514,1538,23188,Germany vs Portugal,2025-06-04 19:10:00,5
13,19330560,1538,23188,Germany vs France,2025-06-08 13:00:00,5
14,19427830,720,21887,Slovakia vs Germany,2025-09-04 18:45:00,5
15,19427850,720,21887,Germany vs Northern Ireland,2025-09-07 18:45:00,5
16,19427884,720,21887,Germany vs Luxembourg,2025-10-10 18:45:00,5
17,19427920,720,21887,Northern Ireland vs Germany,2025-10-13 18:45:00,5


In [8]:
summary = df.groupby("league_id").size().rename("matches").sort_index()
summary

league_id
720      6
1326     4
1538    10
Name: matches, dtype: int64

In [9]:
out_csv = SPORTMONKS_DIR / "fixtures_germany_201.csv"
df.to_csv(out_csv, index=False)
out_csv.resolve()

WindowsPath('C:/Users/Lenovo/Documents/football-analytics/sportmonks/fixtures_germany_201.csv')

### Nagelsmann framing
Julian Nagelsmann took over the Germany senior men’s team before this window; competitive matches in **2024–2025** in the three listed tournaments are interpreted as his-era games for this pipeline (no extra date filter).